In [1]:
%load_ext autoreload

%autoreload 2

In [3]:
from src.ingestion.parser import MarkdownParser

parser = MarkdownParser("data/markdown")

documents = parser.load_documents()

print(documents[0]["province"])
print(documents[0]["content"][:200])

amazonas
# Amazonas: Guía Práctica para el Viajero

La región de Amazonas, ubicada en el norte del Perú, abarca sierra, ceja de selva y selva amazónica, ofreciendo una rica diversidad de paisajes, culturas e h


In [4]:
documents[3]

{'province': 'apurimac',
 'content': '# Apurímac: Guía Práctica para el Viajero\n\nApurímac es una región de la sierra sur del Perú, caracterizada por sus profundos cañones, paisajes andinos impresionantes, rica historia preínca e inca, y una vibrante cultura colonial y viva. Es un destino ideal para los amantes de la naturaleza, la aventura y la cultura.\n\n## 1. Información General\n\n*   **Ubicación:** Sierra sur del Perú.\n*   **Altitud:**\n    *   **Máxima:** 5438 msnm (Huaña)\n    *   **Mínima:** 920 msnm (Pasaje)\n    *   **Capital:** Abancay (2378 msnm)\n*   **Temperatura Promedio:**\n    *   **Máxima:** 23.8 ºC\n    *   **Mínima:** 11.7 ºC\n*   **Clima:**\n    *   **Época de Lluvias:** Diciembre a marzo.\n    *   **Época Seca:** Junio a setiembre.\n    *   **Épocas de Transición:** Marzo a junio y setiembre a diciembre.\n\n### Vías de Acceso\n\n*   **Vía Terrestre:**\n    *   **Desde Lima:**\n        *   Lima-Nasca-Puquio-Abancay: 912 km (aprox. 18 h)\n        *   Lima-Ayacuch

In [12]:
from src.ingestion.chunker import MarkdownChunker
chunker = MarkdownChunker()

chunks = []

for document in documents:

    chunks.extend(
        chunker.split(
            document["province"],
            document["content"],
        )
    )

print(len(chunks))
print(chunks[0])

767
{'id': 'amazonas_0000', 'province': 'amazonas', 'section': None, 'title': None, 'content': 'La región de Amazonas, ubicada en el norte del Perú, abarca sierra, ceja de selva y selva amazónica, ofreciendo una rica diversidad de paisajes, culturas e historia.'}


In [16]:
print(chunks[3])

{'id': 'amazonas_0003', 'province': 'amazonas', 'section': '¿Cómo Llegar?', 'title': 'Vía Terrestre', 'content': 'Las siguientes rutas conducen a Chachapoyas:  \n*   **Lima - Chiclayo - Chachapoyas:** 1219 km / 22 horas\n*   **Trujillo - Chiclayo - Chachapoyas:** 653 km / 13 horas\n*   **Chiclayo - Chachapoyas:** 449 km / 10 horas\n*   **Cajamarca - Chachapoyas:** 336 km / 10 horas\n*   **Jaén - Chachapoyas:** 183 km / 4 horas'}


### SEARCH

#### TEXT SEARCH

In [17]:
from minsearch import Index

In [18]:
def build_index(documents):
    index = Index(
        text_fields=["content", "title", "section", "province"],
        keyword_fields=["id"]
    )
    index.fit(documents)
    return index

In [19]:
index = build_index(chunks)

In [42]:
question = "Machupicchu?"

search_results = index.search(
    question,
    boost_dict={"content": 4.0, "title": 1.5, "section": 1.0, "province": 2.0},
    num_results=5
)

search_results

[{'id': 'cusco_0018',
  'province': 'cusco',
  'section': '¿Qué Conocer?',
  'title': 'Provincia de Urubamba (Valle Sagrado)',
  'content': '*   **Camino Inca:** Parte de la red de Qhapaq Ñan, considerado la mejor ruta de caminata del Perú, ofrece monumentos arqueológicos, paisajes impresionantes y diversidad de flora y fauna. Se inicia en el km 82 (4 días/3 noches) o km 104 (1 día) de la vía férrea Cusco-Machupicchu. Requiere agencia de viajes autorizada.\n*   **Santuario Histórico – Parque Arqueológico Nacional de Machupicchu:** Alberga más de 60 monumentos arqueológicos, siendo Machupicchu el más importante, construido alrededor del 1400 d.C. como centro religioso, político y administrativo inca. (Desde Ollantaytambo 2 h en bus, tren a Machupicchu Pueblo 2 h, bus final 25 min. L-D, 6:00-17:30, boleto anticipado).\n*   **Abra Málaga:** A 4330 msnm, con bosques de árboles de poca altura, ideal para el avistamiento de aves. (A 150 km NO de Cusco, 3 h en auto).'},
 {'id': 'cusco_0005',


In [44]:
question = "Machu picchu?"

search_results = index.search(
    question,
    boost_dict={"content": 4.0, "title": 1.5, "section": 1.0, "province": 2.0},
    num_results=5
)

search_results

[{'id': 'gastronomia_0031',
  'province': 'gastronomia',
  'section': '2. Regiones Turísticas de Perú',
  'title': '2.5. Cusco: Auténtico y Cosmopolita',
  'content': 'Cuna del imperio incaico, Cusco impresiona por la vitalidad de su historia, su arquitectura colonial sobre piedras incas, sitios arqueológicos y una gastronomía que es historia, conocimiento y adaptación.  \n#### 2.5.1. Información General\n*   **Altitud (MSNM)**:\n*   Cusco: 3399 m\n*   Machu Picchu: 2490 m\n*   Chinchero: 3754 m\n*   Urubamba: 2871 m\n*   Ollantaytambo: 2792 m\n*   **Temperaturas**: Mínimas: 1 °C; Máximas: 20 °C.\n*   **Mejor momento para viajar**: Estación seca (abril a octubre).\n*   **Cómo llegar**: Aeropuerto Internacional Velasco Astete.\n*   **Gastronomía**: Basada en productos locales, recetarios regionales e ingeniería agrícola ancestral. Calca produce más de 100 variedades de papas nativas. Moray, un centro de experimentación agrícola inca.'},
 {'id': 'gastronomia_0033',
  'province': 'gastron

#### VECTOR SEARCH

In [43]:
def build_embedding_text(chunk: dict) -> str:
    return f"""
        Provincia: {chunk['province']}

        Sección: {chunk['section']}

        Título: {chunk['title']}

        Contenido:
        {chunk['content']}
        """.strip()

In [48]:
from pathlib import Path

cache = Path("models")

for path in cache.rglob("*"):
    print(path)

models/models--xenova--paraphrase-multilingual-mpnet-base-v2
models/CACHEDIR.TAG
models/models--qdrant--paraphrase-multilingual-MiniLM-L12-v2-onnx-Q
models/.locks
models/models--xenova--paraphrase-multilingual-mpnet-base-v2/refs
models/models--xenova--paraphrase-multilingual-mpnet-base-v2/blobs
models/models--xenova--paraphrase-multilingual-mpnet-base-v2/files_metadata.json
models/models--xenova--paraphrase-multilingual-mpnet-base-v2/trees
models/models--xenova--paraphrase-multilingual-mpnet-base-v2/snapshots
models/models--qdrant--paraphrase-multilingual-MiniLM-L12-v2-onnx-Q/refs
models/models--qdrant--paraphrase-multilingual-MiniLM-L12-v2-onnx-Q/blobs
models/models--qdrant--paraphrase-multilingual-MiniLM-L12-v2-onnx-Q/trees
models/models--qdrant--paraphrase-multilingual-MiniLM-L12-v2-onnx-Q/snapshots
models/.locks/models--xenova--paraphrase-multilingual-mpnet-base-v2
models/.locks/models--qdrant--paraphrase-multilingual-MiniLM-L12-v2-onnx-Q
models/.locks/models--xenova--paraphrase-mu

In [50]:
from fastembed import TextEmbedding

embedding_model = TextEmbedding(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    cache_dir="./models"
)

/tmp/ipykernel_22734/1916916902.py:3: UserWarning: The model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  embedding_model = TextEmbedding(


In [52]:
chunk = chunks[0]
text = build_embedding_text(chunk)
text

'Provincia: amazonas\n\n        Sección: None\n\n        Título: None\n\n        Contenido:\n        La región de Amazonas, ubicada en el norte del Perú, abarca sierra, ceja de selva y selva amazónica, ofreciendo una rica diversidad de paisajes, culturas e historia.'

In [56]:
embedding = list(
    embedding_model.embed([text])
)[0]

In [57]:
embedding.shape

(384,)

In [58]:
texts = []

for chunk in chunks:
    text = build_embedding_text(chunk)
    texts.append(text)

In [59]:
len(texts)

767

In [61]:
texts[10]

'Provincia: amazonas\n\n        Sección: ¿Qué Conocer? (Atractivos Turísticos)\n\n        Título: Provincia de Chachapoyas\n\n        Contenido:\n        *   **Ubicación:** Intersección de jirones Ayacucho y Ortiz Arrieta.\n*   **Horario:** Lunes a Viernes 9:00-13:00 h.\n*   **Descripción:** Lugar de nacimiento del prócer de la Independencia Toribio Rodríguez de Mendoza (17 de abril de 1750). Actualmente sede del Obispado de Chachapoyas. Destacan el zaguán, patio principal, Sala de Obispos y Sala de Recibo con influencia colonial y republicana.\n*   **Casa de las Dos Rosas:**\n*   **Ubicación:** Cruce de los jirones Amazonas y La Merced.\n*   **Horario:** Lunes a Sábado 9:00-12:00 h / 14:30-18:00 h.\n*   **Ingreso:** Con boleto.\n*   **Descripción:** Construcción del siglo XVIII con características arquitectónicas coloniales y republicanas. Destacan su oratorio y sala. Propiedad de la familia Torrejón Montesa desde fines del siglo XIX.\n*   **Plazuela de la Independencia:**\n*   **Ubic

In [62]:
from tqdm.auto import tqdm
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size), desc="Embedding texts"):
    batch = texts[i:i + batch_size]
    batch_vectors = embedding_model.embed(batch)
    vectors.extend(batch_vectors)

len(vectors)

Embedding texts:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding texts: 100%|██████████| 16/16 [00:28<00:00,  1.76s/it]


767

In [63]:
import numpy as np
X = np.array(vectors, dtype=np.float32)
X.shape

(767, 384)

In [64]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["id"])
vindex.fit(X, chunks)

In [80]:
query = "MachuPicchu?"
query_vector = list(embedding_model.embed(query))[0]

In [81]:
query_vector

array([ 2.01734543e-01, -1.57604218e-02,  2.99385071e-01, -1.64036751e-02,
        2.26566315e-01, -6.21490479e-02, -1.26033783e-01,  1.28112793e-01,
       -1.78566933e-01,  6.36329651e-02,  1.73149109e-01, -3.50181580e-01,
        1.91091537e-01,  1.48780346e-02, -3.56109619e-01, -1.06079102e-01,
       -1.80362701e-01, -3.07586670e-01, -1.99710846e-01,  2.11067200e-01,
       -5.80806732e-02, -2.02125549e-01,  2.05119371e-01,  4.94899750e-02,
        4.22607422e-01, -1.40590668e-01, -3.45367432e-01,  1.35512352e-02,
       -5.67135811e-02, -1.36874199e-01,  4.09484863e-01, -2.91458130e-01,
        1.25143051e-01, -1.26716614e-01,  3.06427002e-01,  4.86228943e-01,
       -2.35843658e-03,  8.48617554e-02,  3.33023071e-02,  4.10888672e-01,
       -1.05831146e-01, -7.56835938e-02,  3.15711975e-01,  4.54463959e-02,
        1.38383865e-01, -5.77898026e-02,  4.89821434e-02,  1.61659718e-01,
       -1.46584511e-01,  1.82342529e-03,  4.37297821e-02,  8.69817734e-02,
       -2.36976624e-01, -

In [82]:
results = vindex.search(query_vector, num_results=5)

In [83]:
results

[{'id': 'pasco_0014',
  'province': 'pasco',
  'section': 'Atractivos Turísticos',
  'title': 'Provincia de Oxapampa',
  'content': '*   **Ingreso**: Con boleto.\n*   **Pozuzo**:\n*   **Ubicación**: A 84 km al norte de Oxapampa (3 h en auto).\n*   **Descripción**: El primer lugar donde llegaron los colonos austro-alemanes en 1859. Fundado con un diseño inspirado en sus ciudades natales, con viviendas de plantas geométricas, pisos de madera y techos a dos aguas.\n*   **Atractivos principales**: Capilla San José, monumento al padre José Egg, la Casa de la Cultura, el cementerio de los colonos, las pozas naturales del río Guacamayo y una fábrica de cerveza artesanal.\n*   **Museo Schafferer**:\n*   **Ubicación**: Av. Los Colonos s/n.\n*   **Descripción**: Narra la historia de los colonos austro-alemanes a través de la exhibición de objetos y pertenencias que trajeron consigo, como herramientas, utensilios de cocina, cerámica, fotografías y muebles.\n*   **Horario**: L-S, 9:00-13:00 y 14:3

In [88]:
query =  "Ciudadela inca ubicada en Cusco"
query_vector = list(embedding_model.embed(query))[0]
results = vindex.search(query_vector, num_results=5)
results

[{'id': 'cusco_0006',
  'province': 'cusco',
  'section': '¿Qué Conocer?',
  'title': 'Provincia de Cusco (Ciudad y Alrededores)',
  'content': '#### Atractivos en la Ciudad de Cusco  \n*   **Plaza de Armas:** Antiguo centro ceremonial incaico donde se realizaba el Inti Raymi. Su arquitectura colonial incluye arcos de piedra y construcciones históricas.\n*   **Catedral:** Estilo renacentista, construida sobre el Templo de Suntur Wasi y el Palacio del Inca Wiracocha. Alberga arte de la escuela cusqueña y objetos de plata. (L-D, 10:30-18:30, con boleto).\n*   **Museo de Historia Natural:** Exhibe muestras geológicas y paleontológicas de Cusco, así como especímenes de la diversidad biológica regional, destacando un colmillo de mastodonte. (L-V, 9:00-14:30, con boleto).\n*   **Iglesia de la Compañía de Jesús:** Imponente ejemplar del barroco andino, con portada que simula un retablo y paredes de piedra tallada. (L-D, 9:00-11:45 y 13:00-17:45, con boleto).\n*   **ChocoMuseo Cusco:** Ofrece 

In [ ]:
query =  "Quiero visitar la maravilla del mundo"
query_vector = list(embedding_model.embed(query))[0]
results = vindex.search(query_vector, num_results=5)
results

[{'id': 'lima_callao_0031',
  'province': 'lima_callao',
  'section': 'Atractivos Turísticos por Distrito',
  'title': 'Distrito de Pueblo Libre',
  'content': '*   **Museo Larco:**\n*   **Ubicación:** Av. Bolívar 1515.\n*   **Atención:** L-D 9:00-22:00 h.\n*   **Ingreso:** Con boleto.\n*   **Descripción:** Considerado uno de los 25 mejores museos del mundo, situado en una mansión virreinal del siglo XVIII con hermosos jardines. Exhibe ceramios, objetos de oro, textiles y arte precolombino, incluyendo su colección de huacos eróticos. Su depósito, con 45,000 objetos arqueológicos, es uno de los pocos del mundo abierto al público.\n*   **Complejo Arqueológico Mateo Salado:**\n*   **Ubicación:** Entre las cuadras 12 y 13 de la Av. Mariano Cornejo, a 200 m de plaza de la Bandera.\n*   **Atención:** Mi-D 9:00-16:00 h.\n*   **Ingreso:** Con boleto.\n*   **Descripción:** Centro administrativo y ceremonial construido por los Ychsma (1100-1450 d.C.) con cinco pirámides escalonadas truncas. Fue 